# Airbnb NYC 2019 Sentiment Analysis
## Notebook 1: Data Preprocessing & Exploratory Data Analysis
## Task: Data Preprocessing & Exploratory Data Analysis

**Pipeline:**
1. Load raw reviews dataset
2. Merge with listings for richer features
3. Text cleaning (lowercase â†’ punctuation â†’ stopwords â†’ lemmatize)
4. Sentiment labeling with **VADER**
5. EDA: class distribution, word clouds, bigrams, time trends
6. Save cleaned dataset

In [ ]:
# Libraries
import warnings, re
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from collections import Counter

from wordcloud import WordCloud
from pathlib import Path

# Download NLTK resources
for pkg in ['vader_lexicon', 'stopwords', 'wordnet', 'punkt',
            'averaged_perceptron_tagger', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

# Paths
BASE      = Path('.')
DATASET   = BASE / 'Dataset'
CLEANED   = DATASET / 'Cleaned Data'
CLEANED.mkdir(exist_ok=True)

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': '#0f0f1a',
                     'axes.facecolor': '#1a1a2e', 'axes.labelcolor': 'white',
                     'xtick.color': 'white', 'ytick.color': 'white',
                     'text.color': 'white', 'grid.color': '#2e2e4e',
                     'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = {'Positive': '#00e5ff', 'Neutral': '#ffd54f', 'Negative': '#ff5252'}

print('âœ… Environment ready')

---
### 1. Load Data

In [ ]:
reviews  = pd.read_csv(DATASET / 'reviews.csv', parse_dates=['date'])
listings = pd.read_csv(DATASET / 'AB_NYC_2019.csv')

print(f'Reviews  : {reviews.shape}')
print(f'Listings : {listings.shape}')
reviews.head()

In [ ]:
# Merge for neighbourhood & room_type context
df = reviews.merge(
    listings[['id','neighbourhood_group','neighbourhood','room_type','price']],
    left_on='listing_id', right_on='id', how='left'
).drop(columns='id_y').rename(columns={'id_x': 'review_id'})

# Drop reviews without text
df.dropna(subset=['comments'], inplace=True)
df['comments'] = df['comments'].astype(str)

print(f'Merged dataset: {df.shape}')
df.info()

---
### 2. Text Cleaning

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """Lowercase â†’ remove non-alpha â†’ tokenize â†’ remove stops â†’ lemmatize."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # keep only letters
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

df['cleaned_text'] = df['comments'].apply(clean_text)

# Show sample
pd.set_option('display.max_colwidth', 120)
df[['comments', 'cleaned_text']].sample(5, random_state=42)

---
### 3. Sentiment Labeling with VADER

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text: str) -> str:
    """Map VADER compound score â†’ Positive / Neutral / Negative."""
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:  return 'Positive'
    elif score <= -0.05: return 'Negative'
    else: return 'Neutral'

# Run on original (uncleaned) text for best VADER accuracy
df['compound_score'] = df['comments'].apply(
    lambda t: sia.polarity_scores(t)['compound']
)
df['sentiment'] = df['compound_score'].apply(
    lambda s: 'Positive' if s >= 0.05 else ('Negative' if s <= -0.05 else 'Neutral')
)

print(df['sentiment'].value_counts())
print(df['sentiment'].value_counts(normalize=True).mul(100).round(1))

---
### 4. EDA

In [ ]:
# â”€â”€ 4.1 Sentiment distribution (Plotly) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
counts = df['sentiment'].value_counts().reset_index()
counts.columns = ['sentiment', 'count']

fig = px.bar(
    counts, x='sentiment', y='count', color='sentiment',
    color_discrete_map=PALETTE,
    title='<b>Sentiment Distribution â€“ Airbnb NYC Reviews</b>',
    template='plotly_dark', text='count'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, title_font_size=18)
fig.show()

In [ ]:
# â”€â”€ 4.2 Word Clouds per Sentiment â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f1a')
WC_COLORS  = {'Positive': 'Blues', 'Neutral': 'YlOrBr', 'Negative': 'Reds'}

for ax, sentiment in zip(axes, ['Positive', 'Neutral', 'Negative']):
    corpus = ' '.join(df[df['sentiment'] == sentiment]['cleaned_text'])
    wc = WordCloud(
        width=600, height=400, max_words=80,
        background_color='#1a1a2e',
        colormap=WC_COLORS[sentiment]
    ).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{sentiment} Reviews', fontsize=14, color=PALETTE[sentiment], pad=12)
    ax.axis('off')

plt.suptitle('Word Clouds by Sentiment', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig('Images/wordclouds.png', bbox_inches='tight',
            facecolor='#0f0f1a', dpi=150)
plt.show()

In [ ]:
# â”€â”€ 4.3 Top Bigrams â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from nltk import bigrams

def top_ngrams(texts, n=20):
    all_tokens = ' '.join(texts).split()
    return Counter(bigrams(all_tokens)).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.patch.set_facecolor('#0f0f1a')

for ax, sent in zip(axes, ['Positive', 'Neutral', 'Negative']):
    ngrams_data = top_ngrams(df[df['sentiment'] == sent]['cleaned_text'], n=15)
    labels = [' '.join(b) for b, _ in ngrams_data]
    counts_ = [c for _, c in ngrams_data]
    bars = ax.barh(labels[::-1], counts_[::-1],
                   color=PALETTE[sent], alpha=0.85)
    ax.set_title(f'{sent} â€“ Top Bigrams', color=PALETTE[sent], fontsize=12)
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('Images/bigrams.png', bbox_inches='tight',
            facecolor='#0f0f1a', dpi=150)
plt.show()

In [ ]:
# â”€â”€ 4.4 Sentiment by Neighbourhood Group â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
borough_sent = (
    df.groupby(['neighbourhood_group', 'sentiment'])
    .size().reset_index(name='count')
)
fig = px.bar(
    borough_sent, x='neighbourhood_group', y='count',
    color='sentiment', color_discrete_map=PALETTE, barmode='group',
    title='<b>Sentiment by NYC Borough</b>', template='plotly_dark'
)
fig.update_layout(title_font_size=18, xaxis_title='Borough',
                  yaxis_title='Review Count')
fig.show()

In [ ]:
# â”€â”€ 4.5 VADER Score Distribution â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig = px.histogram(
    df, x='compound_score', color='sentiment',
    color_discrete_map=PALETTE, nbins=60, marginal='box',
    title='<b>VADER Compound Score Distribution</b>', template='plotly_dark'
)
fig.update_layout(title_font_size=18)
fig.show()

In [ ]:
# â”€â”€ 4.6 Reviews over Time â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df['year_month'] = df['date'].dt.to_period('M').astype(str)
time_sent = (
    df.groupby(['year_month', 'sentiment'])
    .size().reset_index(name='count')
)

fig = px.line(
    time_sent, x='year_month', y='count', color='sentiment',
    color_discrete_map=PALETTE,
    title='<b>Review Volume over Time by Sentiment</b>', template='plotly_dark'
)
fig.update_layout(title_font_size=18, xaxis_title='Month',
                  xaxis_tickangle=-45)
fig.show()

---
### 5. Save Cleaned Dataset

In [ ]:
save_cols = ['review_id', 'listing_id', 'date', 'reviewer_name',
             'comments', 'cleaned_text', 'compound_score', 'sentiment',
             'neighbourhood_group', 'neighbourhood', 'room_type', 'price']

df[save_cols].to_csv(CLEANED / 'reviews_cleaned.csv', index=False)
print(f'âœ… Saved to {CLEANED / "reviews_cleaned.csv"} â€” {len(df):,} rows')
df[save_cols].head()